In [39]:
import json, re, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats

from src.compute_benchmarks import run_benchmarks_standalone, compute_metrics as bm_compute_metrics
from data.wrds_data import WRDSDATA

# ── Configuration ─────────────────────────────────────────────────────────
SINGLE_AGENT_DIR = Path("new_results/portfolios_single")   # single-agent JSONs
MULTI_AGENT_DIR = Path("results/portfolios_multi")       # multi-agent JSONs
INITIAL_CAPITAL = 1_000_000.0
PERIODS_PER_YEAR = 4  # quarterly
RF_ANNUAL = 0.02
BENCHMARK = "benchmarks.json"
cmap = plt.cm.get_cmap('tab10')
colors = [cmap(i) for i in np.linspace(0, 1, 10)]

# ── Experiment parameters
START_DATE = "2014-01-01"
END_DATE   = "2023-12-31"
FREQUENCY  = "quarterly"
N_STOCKS   = None          # None = all available stocks

print("Config loaded.")
print(f"Single-agent DIR: {SINGLE_AGENT_DIR.resolve()}")
print(f"Multi-agent DIR: {MULTI_AGENT_DIR.resolve()}")

Config loaded.
Single-agent DIR: C:\git\ASIM-LLM-MAS\new_results\portfolios_single
Multi-agent DIR: C:\git\ASIM-LLM-MAS\results\portfolios_multi


C:\Users\nikla\AppData\Local\Temp\ipykernel_8580\133672102.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab10')


In [40]:
# ── Ticker Selection Analysis ────────────────────────────────────────────────

def analyze_ticker_allocations_single_agent(portfolio_data: dict) -> pd.DataFrame:
    """Analyze ticker allocations across all periods from a single portfolio JSON.
    
    Args:
        portfolio_data: Dictionary from portfolio JSON containing 'snapshots' key
        
    Returns:
        DataFrame with ticker-level statistics:
        - ticker: ticker symbol
        - periods_held: count of periods ticker was held
        - pct_periods_held: percentage of all periods
        - avg_allocation: average weight_pct when held
        - max_allocation: peak weight_pct
        - min_allocation: minimum weight_pct (when held)
        - first_period: first period ticker appears
        - last_period: last period ticker disappears
        - max_consecutive: longest consecutive holding period
    """
    snapshots = portfolio_data.get("snapshots", [])
    if not snapshots:
        return pd.DataFrame()
    
    total_periods = len(snapshots)
    ticker_data = {}  # {ticker: {periods, weights, first, last, consecutive_blocks}}
    
    for period_idx, snapshot in enumerate(snapshots):
        period_name = snapshot.get("period")
        allocation = snapshot.get("allocation", {})
        
        # Extract all tickers (exclude cash)
        for ticker, alloc_info in allocation.items():
            if ticker == "cash":
                continue
                
            weight_pct = alloc_info.get("weight_pct", 0.0)
            
            # Only track tickers with meaningful allocation (> 0.01%)
            if weight_pct < 0.01:
                continue
            
            if ticker not in ticker_data:
                ticker_data[ticker] = {
                    "periods": [],
                    "weights": [],
                    "first_period": period_idx,
                    "last_period": period_idx,
                    "consecutive_blocks": [],
                }
            
            ticker_data[ticker]["periods"].append(period_idx)
            ticker_data[ticker]["weights"].append(weight_pct)
            ticker_data[ticker]["last_period"] = period_idx
    
    # Build result rows
    rows = []
    for ticker, data in ticker_data.items():
        periods_held = len(data["periods"])
        weights = data["weights"]
        
        # Calculate consecutive blocks
        periods = sorted(data["periods"])
        consecutive_blocks = []
        if periods:
            block_start = periods[0]
            block_length = 1
            for i in range(1, len(periods)):
                if periods[i] == periods[i-1] + 1:
                    block_length += 1
                else:
                    consecutive_blocks.append(block_length)
                    block_start = periods[i]
                    block_length = 1
            consecutive_blocks.append(block_length)
        
        row = {
            "ticker": ticker,
            "periods_held": periods_held,
            "pct_periods_held": round(100 * periods_held / total_periods, 2),
            "avg_allocation": round(np.mean(weights), 2),
            "max_consecutive": max(consecutive_blocks) if consecutive_blocks else 1,
            "num_entry_exit": len(consecutive_blocks),
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Sort by frequency of holding
    df = df.sort_values("periods_held", ascending=False).reset_index(drop=True)
    
    return df


def load_and_analyze_single_agent(agent_dir: Path, agent_name: str, run_idx: str = "01") -> tuple[pd.DataFrame, dict]:
    """Load a single agent's portfolio JSON and return allocation analysis.
    
    Args:
        agent_dir: Path to agent portfolio directory
        agent_name: Name of the agent (used to find JSON file)
        run_idx: Which run to load (default "01")
        
    Returns:
        Tuple of (ticker_analysis_df, portfolio_json_dict)
    """
    # Construct filename
    filename = f"{agent_name}/{agent_name}_run{run_idx}.json"
    filepath = agent_dir / filename
    
    if not filepath.exists():
        raise FileNotFoundError(f"Portfolio file not found: {filepath}")
    
    with open(filepath) as f:
        pdata = json.load(f)
    
    df_tickers = analyze_ticker_allocations_single_agent(pdata)
    
    # Add metadata
    df_tickers["agent"] = agent_name
    df_tickers["run"] = run_idx
    
    return df_tickers, pdata


# ── Test with a single agent ──────────────────────────────────────────────
print("=" * 80)
print("TICKER ALLOCATION ANALYSIS - SINGLE AGENT")
print("=" * 80)

# Analyze Joel Greenblatt (a known agent)
agent_name = "joel_greenblatt"
run_idx = "01"

df_ticker_analysis, portfolio_json = load_and_analyze_single_agent(
    SINGLE_AGENT_DIR, agent_name, run_idx
)

print(f"\nAgent: {agent_name} (Run {run_idx})")
print(f"Total unique tickers held (>0.01%): {len(df_ticker_analysis)}")
print(f"\nTop 15 most-held tickers:\n")
print(df_ticker_analysis.head(15).to_string(index=False))

TICKER ALLOCATION ANALYSIS - SINGLE AGENT

Agent: joel_greenblatt (Run 01)
Total unique tickers held (>0.01%): 9

Top 15 most-held tickers:

     ticker  periods_held  pct_periods_held  avg_allocation  max_consecutive  num_entry_exit           agent run
TICK_48C6EE            31             75.61           94.33               31               1 joel_greenblatt  01
TICK_4AF927            19             46.34            0.04               19               1 joel_greenblatt  01
TICK_F716FF            14             34.15            0.01                7               4 joel_greenblatt  01
TICK_0E60CD             8             19.51           50.92                8               1 joel_greenblatt  01
TICK_4D3913             4              9.76           47.33                4               1 joel_greenblatt  01
TICK_55F196             4              9.76           39.32                4               1 joel_greenblatt  01
TICK_8D420B             3              7.32            0.05         

In [41]:
# ── Aggregate ticker analysis across all runs and agents ────────────────────

def aggregate_ticker_analysis_all_runs(portfolio_dir: Path) -> pd.DataFrame:
    """Analyze ticker allocations for ALL agents and ALL runs.
    
    Iterates through portfolio directory, finds all agent/run combinations,
    and aggregates ticker allocation statistics into one big dataframe.
    
    Args:
        portfolio_dir: Path to portfolios directory
        
    Returns:
        DataFrame with columns:
        - agent, run, ticker, periods_held, pct_periods_held, 
        - avg_allocation, max_consecutive, num_entry_exit
    """
    portfolio_files = sorted(portfolio_dir.glob("**/*.json"))
    
    if not portfolio_files:
        raise ValueError(f"No portfolio files found in {portfolio_dir}")
    
    all_results = []
    file_count = 0
    
    print(f"Processing {len(portfolio_files)} portfolio files...\n")
    
    for fp in portfolio_files:
        try:
            with open(fp) as f:
                pdata = json.load(f)
        except (json.JSONDecodeError, IOError) as e:
            print(f"  Warning: Failed to load {fp.name}: {e}")
            continue
        
        # Extract agent name and run from filename
        # Pattern: agent_name_runXX.json
        stem = fp.stem
        m = re.match(r"(.+?)_run(\d+)$", stem)
        
        if not m:
            print(f"  Warning: Could not parse filename {fp.name}")
            continue
        
        agent_name = m.group(1)
        run_idx = m.group(2)
        
        # Analyze this portfolio
        df_tickers = analyze_ticker_allocations_single_agent(pdata)
        
        if len(df_tickers) > 0:
            # Add metadata columns
            df_tickers["agent"] = agent_name
            df_tickers["run"] = run_idx
            
            all_results.append(df_tickers)
            file_count += 1
        else:
            print(f" {agent_name:20s} run={run_idx:2s}  (no tickers found)")
    
    print(f"\nSuccessfully processed {file_count} portfolios\n")
    
    # Concatenate all results
    df_combined = pd.concat(all_results, ignore_index=True)
    
    return df_combined


# ── Run aggregation ──────────────────────────────────────────────────────────
print("=" * 80)
print("AGGREGATING TICKER ANALYSIS - ALL AGENTS & ALL RUNS")
print("=" * 80)

df_all_tickers = aggregate_ticker_analysis_all_runs(SINGLE_AGENT_DIR)

# Show sample data
print("Sample data (first 3 rows):")
print(df_all_tickers.head(3).to_string(index=False))

AGGREGATING TICKER ANALYSIS - ALL AGENTS & ALL RUNS
Processing 150 portfolio files...


Successfully processed 150 portfolios

Sample data (first 3 rows):
     ticker  periods_held  pct_periods_held  avg_allocation  max_consecutive  num_entry_exit      agent run
TICK_F716FF            38             92.68            1.94               35               2 ben_graham  01
TICK_00BB14            37             90.24           94.46               36               2 ben_graham  01
TICK_0E60CD             5             12.20           52.56                5               1 ben_graham  01


In [42]:
# ── Load reverse ticker map and deanonymize ─────────────────────────────────

print("=" * 80)
print("DEANONYMIZING TICKERS")
print("=" * 80)

# Load reverse ticker map from backtest data
backtest_data_path = Path("private_results/data/backtest_data.json")

if backtest_data_path.exists():
    with open(backtest_data_path) as f:
        backtest_data = json.load(f)
    
    reverse_ticker_map = backtest_data.get("reverse_ticker_map", {})
    
    print(f"\nLoaded reverse_ticker_map with {len(reverse_ticker_map)} mappings")
    
    # Apply mapping to all tickers in the aggregated dataframe
    df_all_tickers["real_ticker"] = df_all_tickers["ticker"].map(reverse_ticker_map)
    
    # Count how many were successfully mapped
    mapped_count = df_all_tickers["real_ticker"].notna().sum()
    unmapped_count = df_all_tickers["real_ticker"].isna().sum()
    
    print(f"Mapping results:")
    print(f"  Successfully mapped: {mapped_count} tickers")
    print(f"  Unmapped (not in reverse_map): {unmapped_count} tickers")
    print()
    
    # Show sample of mapped data
    print("Sample data with real tickers:\n")
    display_cols = ["ticker", "real_ticker", "agent", "periods_held", "avg_allocation"]
    print(df_all_tickers[display_cols].head(4).to_string(index=False))
    
else:
    print(f"\nWarning: backtest_data.json not found at {backtest_data_path}")
    print("Cannot load reverse_ticker_map")

print("\n")

DEANONYMIZING TICKERS

Loaded reverse_ticker_map with 186 mappings
Mapping results:
  Successfully mapped: 904 tickers
  Unmapped (not in reverse_map): 0 tickers

Sample data with real tickers:

     ticker real_ticker      agent  periods_held  avg_allocation
TICK_F716FF        AAPL ben_graham            38            1.94
TICK_00BB14        AVGO ben_graham            37           94.46
TICK_0E60CD          MU ben_graham             5           52.56
TICK_5FE955        ISRG ben_graham             5            9.45




In [65]:
# ── GroupBy Analysis with Real Tickers ───────────────────────────────

print("=" * 80)
print("GROUPBY ANALYSIS - WITH REAL TICKER NAMES")
print("=" * 80)

# 1. Most frequently held tickers across all agents/runs
ticker_frequency_real = df_all_tickers.groupby("real_ticker").agg({
    "periods_held": ["mean", "max", "count"],
    "pct_periods_held": "mean",
    "avg_allocation": "mean",
    "num_entry_exit": "mean",
    "agent": "nunique"
}).round(2)
ticker_frequency_real.columns = ["avg_periods_held", "max_periods_held", "count_appearances", 
                                  "avg_pct_periods", "avg_allocation_pct", "avg_num_trades", "num_agents"]
ticker_frequency_real = ticker_frequency_real.sort_values("count_appearances", ascending=False)

GROUPBY ANALYSIS - WITH REAL TICKER NAMES


In [66]:
print("\n MOST FREQUENTLY HELD TICKERS (across all agents & runs):")
# Reorder columns - put num_agents first, then by count_appearances, etc.
ticker_frequency_real = ticker_frequency_real[["count_appearances", "num_agents", "avg_periods_held", 
                                                "max_periods_held", "avg_allocation_pct", 
                                                "avg_pct_periods", "avg_num_trades"]]
ticker_frequency_real.head(20)


 MOST FREQUENTLY HELD TICKERS (across all agents & runs):


,count_appearances,num_agents,avg_periods_held,max_periods_held,avg_allocation_pct,avg_pct_periods,avg_num_trades
real_ticker,,,,,,,
MU,104,5,8.87,36,77.82,21.62,1.24
NVDA,103,5,18.17,41,51.46,44.31,1.38
AVGO,80,5,14.96,41,59.45,36.49,1.32
AAPL,54,5,27.57,41,66.19,67.25,1.09
ISRG,38,5,17.37,41,60.91,42.36,1.05
GOOGL,33,5,19.94,38,44.74,48.63,1.03
INTC,27,5,12.07,35,32.90,29.45,1.11
LRCX,25,5,12.08,29,20.01,29.46,1.20
NTES,21,5,13.29,26,11.68,32.40,1.19


In [47]:
# Agent comparison: which agents favor which holding strategies?
print("\n\n AGENT COMPARISON (average holding patterns):")
agent_summary_real = df_all_tickers.groupby("agent").agg({
    "periods_held": "mean",
    "max_consecutive": "mean",
    "num_entry_exit": "mean",
    "avg_allocation": "mean",
}).round(2)
agent_summary_real.columns = ["avg_hold_periods", "avg_max_consecutive", "avg_num_trades", "avg_position_size"]
agent_summary_real = agent_summary_real.sort_values("avg_hold_periods", ascending=False)
print(agent_summary_real)



 AGENT COMPARISON (average holding patterns):
                 avg_hold_periods  avg_max_consecutive  avg_num_trades  \
agent                                                                    
ben_graham                  13.97                13.39            1.16   
cathie_wood                 12.55                12.20            1.11   
ray_dalio                   11.71                11.13            1.18   
buffett                     10.63                10.23            1.11   
joel_greenblatt              9.87                 9.36            1.14   

                 avg_position_size  
agent                               
ben_graham                   51.08  
cathie_wood                  28.86  
ray_dalio                    38.32  
buffett                      43.37  
joel_greenblatt              42.93  


In [48]:
# ── Ticker-Specific Analysis ──────────────────────────────────────

print("=" * 80)
print("TICKER-SPECIFIC ANALYSIS")
print("=" * 80)

# Analyze a specific ticker across all agents
ticker_of_interest = "AAPL"
print(f"\n1. TICKER: {ticker_of_interest}")
print(f"   Which agents bought this ticker? How often? With what size?\n")

aapl_data = df_all_tickers[df_all_tickers["real_ticker"] == ticker_of_interest].copy()
if len(aapl_data) > 0:
    aapl_summary = aapl_data.groupby("agent").agg({
        "periods_held": ["min", "mean", "max"],
        "avg_allocation": ["min", "mean", "max"],
        "num_entry_exit": "mean",
        "run": "count"
    }).round(2)
    aapl_summary.columns = ["hold_min", "hold_mean", "hold_max", 
                             "alloc_min", "alloc_mean", "alloc_max", "trades", "appearances"]
    print(aapl_summary)
else:
    print(f"   {ticker_of_interest} not found in any portfolio")

TICKER-SPECIFIC ANALYSIS

1. TICKER: AAPL
   Which agents bought this ticker? How often? With what size?

                 hold_min  hold_mean  hold_max  alloc_min  alloc_mean  \
agent                                                                   
ben_graham             19      28.25        38       1.94       52.51   
buffett                15      29.13        37       0.06       80.60   
cathie_wood            12      26.50        41       0.16       44.10   
joel_greenblatt         5      24.50        38       0.01       50.47   
ray_dalio              20      28.67        33       0.02       76.69   

                 alloc_max  trades  appearances  
agent                                            
ben_graham          100.00    1.12            8  
buffett             100.00    1.00           15  
cathie_wood          88.03    1.00            2  
joel_greenblatt     100.00    1.29           14  
ray_dalio           100.00    1.00           15  
